In this notebook we'll try out estimation of a transfer function using both time-domain and frequency domain data to see how they behave compared to each other. We start by loading the relevant packages. 

In [ ]:
Pollis.packages("ControlSystems", "ControlSystemIdentification", "Plots")

In [ ]:
using ControlSystems, ControlSystemIdentification, Plots

Next, we define the system to be estimated
$$G_{test}(s) = \dfrac{12}{1.1s^2 + 0.0009s + 12}$$
This is a very lightly damped (resonant), second-order system.
We also discretize the system using ZoH and simulate some data from it with a Gaussian input $u$.

In [ ]:
Gtest = tf([12], [1.1, 0.0009, 12]) # True system
h     = 0.01                        # Sample time
u     = randn(1, 10000)
Gd    = c2d(Gtest, h)
y,x,t = lsim(ss(Gd), u);

Next, we construct an `IdData` object and estimate the transfer function using a spectral method

In [ ]:
d = iddata(y, u, h)
H = tfest(d, 0.05)[1]

The spectral method yields a nonparametric estimate of the transfer function, i.e., the value of the transfer function for each frequency in a frequency vector. To obtain a parametric (rational) transfer function, we now call the function `arx` to estimate a (discrete time) model from time-domain data and the function `tfest` to obtain a model from the frequency-domain data. We use the function `d2c` to convert the estimated ARX model from discrete time to continuous time to allow us to compare the estimated poles.

In [ ]:
G0 = tf([10], [1.0, 0.01, 10]) # initial guess
Gf = tfest(H, G0)              # Provide an initial guess for frequency domain
Gt = arx(d, 2, 1) |> d2c       # Provide the model order for time domain
Gf, Gt

We can inspect the error in the estimated poles of the system, and how well the static gain is estimated:

In [ ]:
norm(pole(Gt)-pole(Gtest)), norm(pole(Gf)-pole(Gtest))

In [ ]:
dcgain(Gtest)[], dcgain(Gt)[], dcgain(Gf)[]

It appears as if the time-domain method is better at locating the poles, which is expected since AR-model estimation is considered a spectral-estimation method with "super resolution". The static gain is better estimated by the frequency-domain method.

We can also plot the bode curves of the systems:

In [ ]:
plot(H, lab="Nonparametric", legend=true)
bodeplot!(Gtest, H.w[2:end], lab="True", plotphase=false, l=(3,:dash))
bodeplot!(Gt,    H.w[2:end], lab="Time", plotphase=false, l=(2,))
bodeplot!(Gf,    H.w[2:end], lab="Freq", plotphase=false, l=(2,))